# Advance Crime Data Pipeline

## Stage 1: Ingestion Layer

**Stakeholder:** Police Force Analytics Unit <br>
**Stage:** 1 of 5: Ingestion <br>
**Medallion Layer:** Bronze 🥉 <br>
**Police Forces:** West Midlands · Thames Valley · Surrey · Dyfed-Powys <br>
**Authors:** Group 1 - Adam, Fatima, Josh, Tariq, and Thadsha <br>
**Last Updated:** 18 May 2026 

---
### Purpose

This notebook forms the **Bronze layer** of the medallion pipeline. Its responsibility is to land raw crime data from the Snowflake internal stage into a Bronze table.

---

### How to Use

Run all cells in order from top to bottom. The notebook is designed to be fully repeatable, and no manual steps are required between cells.

**Adding new months**  
Download the new monthly files from `data.police.uk`, unzip the file, and upload them to `@CRIME_PIPELINE.RAW.CRIME_STAGE`. The CSV files should follow the following naming convention: `YYYY-MM-police-force-street`. Re-run the notebook from Section 2 onwards and the file discovery step will automatically detect the new month and include it in the ingestion loop.

**Adding a new force**  
Add the force's filename fragment to `SELECTED_FORCES` in Section 1:  
```python
SELECTED_FORCES = [
    "west-midlands",
    "thames-valley",
    "surrey",
    "dyfed-powys",
    "new-force"   # add here
]
```
Then upload the corresponding CSV files to the stage and re-run the notebook.

---

### Flowchart

Stage Files → File Discovery → Test Batch Validation → Full Batch Ingestion → Validation Checks → Bronze Table Export


## 1. Environment Setup

In [ ]:
%%sql -r dataframe_1
-- Create a warehouse for compute
CREATE WAREHOUSE IF NOT EXISTS CRIME_WH
  WAREHOUSE_SIZE = 'X-SMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE;

-- Create the database for full pipeline
CREATE DATABASE IF NOT EXISTS CRIME_PIPELINE;

-- Create schemas for each pipeline stage
CREATE SCHEMA IF NOT EXISTS CRIME_PIPELINE.RAW; -- Bronze
CREATE SCHEMA IF NOT EXISTS CRIME_PIPELINE.CLEAN; -- Silver
CREATE SCHEMA IF NOT EXISTS CRIME_PIPELINE.REPORTING; -- Gold

-- Create stage for raw CSV uploads
CREATE STAGE IF NOT EXISTS CRIME_PIPELINE.RAW.CRIME_STAGE;


In [ ]:
import pandas as pd
from snowflake.connector.pandas_tools import write_pandas
from snowflake.snowpark.context import get_active_session

session = get_active_session()

# Stage and destination table references
STAGE        = "@CRIME_PIPELINE.RAW.CRIME_STAGE"
BRONZE_TABLE = "CRIME_PIPELINE.RAW.BRONZE_CRIME_RAW"

SELECTED_FORCES = [
    "west-midlands",
    "thames-valley",
    "surrey",
    "cumbria"
    # Add new forces here
]

# Confirm session context is pointing at the correct database and schema
print("Session database :", session.get_current_database())
print("Session schema   :", session.get_current_schema())

## 2. File Discovery

Use Snowflake’s LIST command to query the stage and generate a file inventory. Files are filtered to the four selected forces, with months parsed directly from the filenames.

This method is dynamic, allowing any newly uploaded monthly files are automatically included in the next run with no changes required to the code.

In [ ]:
# LIST returns one row per file in the stage with columns: name, size, md5, last_modified
# Column names are returned with surrounding quotes -- strip them before use
list_df = session.sql(f"LIST {STAGE}").to_pandas()
list_df.columns = [c.strip().strip('"').lower() for c in list_df.columns]

# Retain only files belonging to the four selected forces
list_df["stem"] = list_df["name"].str.split("/").str[-1]
list_df = list_df[
    list_df["stem"].apply(lambda f: any(force in f for force in SELECTED_FORCES))
].reset_index(drop=True)

# Derive month from filename prefix e.g. '2026-01-surrey-street.csv' -> '2026-01'
list_df["month"] = list_df["stem"].str[:7]

months = sorted(list_df["month"].unique())

print(f"{len(list_df)} file(s) identified across {len(months)} month(s):")
print(list_df[["month", "stem", "size"]].to_string(index=False))

## 3. Test Batch

Before processing all available months, the ingestion logic is validated against a single month. This confirms that file reads and column structure are all working correctly before committing to the full run.

In [ ]:
# Use the first available month as the test batch

TEST_MONTHS = [months[0]]  # first available month only

batch_frames = []

for month in TEST_MONTHS:
    month_files = list_df[list_df["month"] == month]["stem"].tolist()

    for filename in month_files:

        try:
            df = pd.read_csv(
                session.file.get_stream(f"{STAGE}/{filename}"),
                dtype=str,
                low_memory=False
            )

            df["source_month"] = month
            df["source_file"]  = filename

            batch_frames.append(df)

            print(f"Loaded: {filename} — {len(df):,} rows")

        except Exception as e:
            print(f"Failed to load {filename}: {e}")
            continue

crime_raw = pd.concat(batch_frames, ignore_index=True)

print(
    f"\nTest batch complete: "
    f"{len(crime_raw):,} rows, "
    f"{crime_raw['source_file'].nunique()} file(s)"
)

## 4. Test Batch Inspection

Validate the structure and content of the test batch before proceeding. Confirms row counts, column names, and that provenance columns have been injected correctly.

In [ ]:
# Inspecting the data
print("Total rows   :", len(crime_raw))
print("Files loaded :", crime_raw["source_file"].nunique())
print("Months loaded:", crime_raw["source_month"].nunique())

crime_raw.head()

## 5. Full Ingestion

With the test batch confirmed, re-run ingestion across all months currently present in the stage. The month list was derived dynamically in Section 2, meaning no manual updates are required when new months are added to the stage.

In [ ]:
batch_frames = []

for month in months:
    month_files = list_df[list_df["month"] == month]["stem"].tolist()

    for filename in month_files:
        df = pd.read_csv(
            session.file.get_stream(f"{STAGE}/{filename}"),
            dtype=str,
            low_memory=False
        )

        df["source_month"] = month
        df["source_file"]  = filename

        batch_frames.append(df)
        print(f"Loaded: {filename} — {len(df):,} rows")

crime_raw = pd.concat(batch_frames, ignore_index=True)

print(f"\nFull ingestion complete: {len(crime_raw):,} rows across {crime_raw['source_month'].nunique()} month(s)")

## 6. Post-Ingestion Validation

Confirm row counts, file coverage, and month completeness before writing to the Bronze table. The row counts recorded here serve as the **baseline** for the Silver cleaning layer's reconciliation report. Any rows removed during cleaning will be compared against these figures.

In [ ]:
print("Total rows   :", len(crime_raw))
print("Files loaded :", crime_raw["source_file"].nunique())
print("Months loaded:", crime_raw["source_month"].nunique())

# Verify all months have loaded
print("\nRows per month:")
print(crime_raw["source_month"].value_counts().sort_index())

# Verify all four forces are present in each month
print("\nRows per force file:")
print(crime_raw["source_file"].value_counts())

## 7. Export Ingested Data to Bronze Table

Silver notebook reads exclusively from this table, not from the stage directly.

The table is overwritten on each run, ensuring Bronze always reflects the current contents of the stage. A read-back verification confirms the written row count matches what was ingested.

In [ ]:
# Reset index before writing to suppress non-standard index warning
crime_raw = crime_raw.reset_index(drop=True)

success, nchunks, nrows, _ = write_pandas(
    conn=session.connection,
    df=crime_raw,
    table_name="BRONZE_CRIME_RAW",  # table name only -- database and schema passed separately below
    database="CRIME_PIPELINE",
    schema="RAW",
    auto_create_table=True,
    overwrite=True
)

print(f"Bronze table written successfully")
print(f"Table  : CRIME_PIPELINE.RAW.BRONZE_CRIME_RAW")
print(f"Rows   : {nrows:,}")
print(f"Chunks : {nchunks}")

In [ ]:
# Read back from the Bronze table to verify the write was successful
# This is the same query the Silver notebook will use as its starting point
crime_raw = session.table(BRONZE_TABLE).to_pandas()    
print("Shape            :", crime_raw.shape)
print("\n-- Column names --")
print(crime_raw.columns.tolist())
print("\n-- Null counts --")
print(crime_raw.isnull().sum())
print("\n-- Sample rows --")
crime_raw.head()